In [1]:
import pandas as pd
import random
import json
import statistics
import numpy as np
from tqdm import tqdm

from utils.load_umls import UMLS
from collections import defaultdict

In [4]:
import os

current_dir = os.getcwd()
if current_dir.endswith('src'):
    os.chdir('..')

print("Current directory:", os.getcwd())

Current directory: /Users/soulofshadow/Developer/UMLS


In [3]:
# 读取 full triplets.csv 文件
triplets_df = pd.read_csv('data/UMLS/2025Full_umls.csv', index_col=0)

cui1_list = triplets_df['cui_1'].tolist()
cui2_list = triplets_df['cui_2'].tolist()
cui_list = list(set(cui1_list + cui2_list))

print("Building adjacency index...")
adj = defaultdict(set)
for cui1, cui2 in zip(triplets_df["cui_1"], triplets_df["cui_2"]):
    adj[cui1].add(cui2)
    adj[cui2].add(cui1)
print(f"Index built: {len(adj)} nodes, {len(triplets_df)} edges")
print(f"{len(triplets_df)}")

Building adjacency index...
Index built: 3283407 nodes, 26439382 edges
26439382


In [5]:
# 读取 DRAGON 的 umls.csv（无header，格式: relation, cui1, cui2）
dragon_df = pd.read_csv("data/UMLS/dragon_umls.csv", header=None, names=["cui_1", "cui_2", "rel"], sep="\t")

cui1_list = dragon_df["cui_1"].tolist()
cui2_list = dragon_df["cui_2"].tolist()
cui_list = list(set(cui1_list + cui2_list))

print("Building adjacency index...")
adj = defaultdict(set)
for cui1, cui2 in zip(dragon_df["cui_1"], dragon_df["cui_2"]):
    adj[cui1].add(cui2)
    adj[cui2].add(cui1)
print(f"Index built: {len(adj)} nodes, {len(dragon_df)} edges")

Building adjacency index...
Index built: 297927 nodes, 1212586 edges


# UMLS size growth by K (using pubmedQA)

In [6]:
def get_dragon_style_khop(seed_entities, max_hop=5):
    seed = set(seed_entities)
    results = {}

    frontier = set(seed)
    all_visited = set(seed)  # 只包含 seed + 被纳入的bridge节点

    for hop in range(1, max_hop + 1):
        # 找与当前frontier相邻的新节点
        next_frontier = set()
        for node in frontier:
            for neighbor in adj[node]:
                if neighbor not in all_visited:
                    next_frontier.add(neighbor)

        # bridge条件：与至少2个原始seed相邻
        bridge_nodes = set()
        for node in next_frontier:
            seed_neighbors = adj[node] & all_visited  # 与当前所有已访问节点相邻的seed节点
            if len(seed_neighbors) >= 2:
                bridge_nodes.add(node)

        # ← 关键修正：只把bridge节点加入all_visited，不是全部next_frontier
        all_visited |= bridge_nodes
        frontier = bridge_nodes  # 下一层从bridge节点继续扩展

        # 统计边数（只在seed + bridge节点之间）
        V = all_visited
        edge_count = sum(1 for u in V for v in adj[u] if v in V and u < v)

        results[f"hop{hop}"] = {
            "cumulative_nodes": len(all_visited),  # seed + 所有bridge节点
            "bridge_nodes": len(all_visited) - len(seed),  # 只有被纳入的bridge节点数
            "cumulative_edges": edge_count,
            "bridge_ratio": (len(all_visited) - len(seed)) / len(seed)
            if seed
            else 0,
        }

        # print(f"  hop-{hop}: {results[f'hop{hop}']}")

    return results


In [13]:
def get_results_for_each_k(test_data):
    # change hop depth（1-4）
    results_exp1 = defaultdict(
        lambda: {"cumulative_nodes": [], "bridge_nodes": [], "cumulative_edges": []}
    )
    for i in tqdm(range(len(test_data))):
        instance = test_data[i]
        if instance.get("concept"):
            cuis = instance["concept"] or []
        elif instance.get("qc"):
            cuis = instance["qc"] or []
        else:
            continue
        r = get_dragon_style_khop(cuis, max_hop=4)
        for hop in range(1, 5):
            stats = r[f"hop{hop}"]
            results_exp1[hop]["cumulative_nodes"].append(stats["cumulative_nodes"])
            results_exp1[hop]["bridge_nodes"].append(stats["bridge_nodes"])
            results_exp1[hop]["cumulative_edges"].append(stats["cumulative_edges"])

    print(
        f"\n{'Hop':<6} {'Cum. Nodes (mean±std)':<25} {'Bridge Nodes (mean±std)':<25} {'Cum. Edges (mean±std)'}"
    )
    print("-" * 85)
    for hop in range(1, 5):
        n = results_exp1[hop]["cumulative_nodes"]
        b = results_exp1[hop]["bridge_nodes"]
        e = results_exp1[hop]["cumulative_edges"]
        print(
            f"  {hop+1:<4} {np.mean(n):.0f} ± {np.std(n):.0f}{'':>10} "
            f"{np.mean(b):.0f} ± {np.std(b):.0f}{'':>10} "
            f"{np.mean(e):.0f} ± {np.std(e):.0f}"
        )

In [9]:
# pubmedQA test - Metamap mapped
with open("data/pubmedQA/metamap_mapped/test.jsonl", "r") as file:
    test_data = [json.loads(line) for line in file]

get_results_for_each_k(test_data)

100%|██████████| 100/100 [00:23<00:00,  4.30it/s]


Hop    Cum. Nodes (mean±std)     Bridge Nodes (mean±std)   Cum. Edges (mean±std)
-------------------------------------------------------------------------------------
  2    193 ± 149           119 ± 144           456 ± 681
  3    1647 ± 1915           1573 ± 1912           13000 ± 19913
  4    11145 ± 9933           11071 ± 9926           94897 ± 85237
  5    46719 ± 31905           46645 ± 31898           299594 ± 191151


In [14]:
# PubmedQA test - Spacy grounded
with open("data/pubmedQA/spacy_grounded/test.grounded.jsonl", "r") as file:
    test_data = [json.loads(line) for line in file]

get_results_for_each_k(test_data)

100%|██████████| 300/300 [01:04<00:00,  4.62it/s]


Hop    Cum. Nodes (mean±std)     Bridge Nodes (mean±std)   Cum. Edges (mean±std)
-------------------------------------------------------------------------------------
  2    182 ± 140           113 ± 135           407 ± 580
  3    1578 ± 1792           1509 ± 1788           12365 ± 18532
  4    10693 ± 9087           10624 ± 9080           93711 ± 80172
  5    46560 ± 30115           46491 ± 30107           301624 ± 180880


# MedQA size growth by changing connectivity

In [16]:
with open('data/medQA/metamap_mapped/test.jsonl', 'r') as file:
    test_data = [json.loads(line) for line in file]

In [18]:
test_data = random.sample(test_data, 10)

In [20]:
def get_unified_khop(seed_entities, max_hop=2):
    seed_entities = [n for n in seed_entities if n in cui_list] 
    seed = set(seed_entities)

    frontier = set(seed)
    all_visited = set(seed)
    results = {}

    for hop in range(1, max_hop + 1):
        next_frontier = set()
        for node in frontier:
            for neighbor in adj[node]:
                if neighbor not in all_visited:
                    next_frontier.add(neighbor)

        all_visited |= next_frontier
        frontier = next_frontier

        # bridge条件：与至少2个原始seed相邻
        bridge_nodes = set()
        for node in next_frontier:
            seed_neighbors = (
                adj[node] & all_visited
            )  # 与当前所有已访问节点相邻的seed节点
            if len(seed_neighbors) >= 2:
                bridge_nodes.add(node)

        # ← 关键修正：只把bridge节点加入all_visited，不是全部next_frontier
        all_visited |= bridge_nodes
        frontier = bridge_nodes  # 下一层从bridge节点继续扩展

        # 只统计 all_visited 内部的边
        V = all_visited
        edge_count = sum(1 for u in V for v in adj[u] if v in V and u < v)

        results[hop + 1] = {
            "cumulative_nodes": len(all_visited),  # seed + 所有bridge节点
            "bridge_nodes": len(all_visited) - len(seed),  # 只有被纳入的bridge节点数
            "cumulative_edges": edge_count,
        }

    return results


def get_qa_khop(q_entities, a_entities, max_hop=2):

    q_entities = [n for n in q_entities if n in cui_list]
    a_entities = [n for n in a_entities if n in cui_list]

    seed_q = set(q_entities)
    seed_a = set(a_entities)

    q_side = set(seed_q)  # 从Q侧可达的所有已纳入节点
    a_side = set(seed_a)  # 从A侧可达的所有已纳入节点
    all_visited = seed_q | seed_a
    frontier = set(all_visited)
    results = {}
    
    for hop in range(1, max_hop + 1):
        candidates = set()
        for node in frontier:
            for neighbor in adj[node]:
                if neighbor not in all_visited:
                    candidates.add(neighbor)

        bridge_nodes = set()
        new_q_side = set()
        new_a_side = set()
        for node in candidates:
            neighbors = adj[node]
            touches_q = bool(neighbors & q_side)
            touches_a = bool(neighbors & a_side)
            if touches_q:
                new_q_side.add(node)  # 这个节点从Q侧可达
            if touches_a:
                new_a_side.add(node)  # 这个节点从A侧可达
            if touches_q and touches_a:
                bridge_nodes.add(node)  # 同时连接Q侧和A侧 → 纳入

        q_side |= new_q_side
        a_side |= new_a_side
        all_visited |= bridge_nodes
        frontier = bridge_nodes

        edge_count = sum(
            1 for u in all_visited for v in adj[u] if v in all_visited and u < v
        )

        results[hop + 1] = {
            "cumulative_nodes": len(all_visited),
            "bridge_nodes": len(all_visited) - len(seed_q | seed_a),
            "cumulative_edges": edge_count,
        }

    return results

In [21]:
# full umls

unified_stats = defaultdict(lambda: defaultdict(list))
qa_only_stats = defaultdict(lambda: defaultdict(list))

for instance in tqdm(test_data):
    q_cuis = instance["concept"] or []
    options_cuis = [instance.get(f"option{i}") or [] for i in range(4)]
    all_option_cuis = [cui for opt in options_cuis for cui in opt]

    # Unified:
    seed_unified = set(q_cuis) | set(all_option_cuis)
    if seed_unified:
        r = get_unified_khop(seed_unified, max_hop=2)
        for hop in [2, 3]:
            unified_stats[hop]["nodes"].append(r[hop]["cumulative_nodes"])
            unified_stats[hop]["bridge"].append(r[hop]["bridge_nodes"])
            unified_stats[hop]["edges"].append(r[hop]["cumulative_edges"])

    # Q->A only:
    if q_cuis and all_option_cuis:
        r = get_qa_khop(q_cuis, all_option_cuis, max_hop=2)
        for hop in [2, 3]:
            qa_only_stats[hop]["nodes"].append(r[hop]["cumulative_nodes"])
            qa_only_stats[hop]["bridge"].append(r[hop]["bridge_nodes"])
            qa_only_stats[hop]["edges"].append(r[hop]["cumulative_edges"])

# 打印结果
print(
    f"{'Hop':<6} {'Strategy':<12} {'Nodes (mean±std)':<25} {'Bridge (mean±std)':<25} {'Edges (mean±std)'}"
)
print("-" * 95)
for hop in [2, 3]:
    for label, stats in [("Unified", unified_stats), ("Q->A only", qa_only_stats)]:
        n = stats[hop]["nodes"]
        b = stats[hop]["bridge"]
        e = stats[hop]["edges"]
        print(
            f"  {hop:<4} {label:<12} "
            f"{np.mean(n):.0f} ± {np.std(n):.0f}{'':>8} "
            f"{np.mean(b):.0f} ± {np.std(b):.0f}{'':>8} "
            f"{np.mean(e):.0f} ± {np.std(e):.0f}"
        )

    u_n = np.mean(unified_stats[hop]["nodes"])
    q_n = np.mean(qa_only_stats[hop]["nodes"])
    u_e = np.mean(unified_stats[hop]["edges"])
    q_e = np.mean(qa_only_stats[hop]["edges"])
    print(
        f"  {hop:<4} {'Ratio':<12} {u_n / q_n:.1f}x{'':>22} {'':>24} {u_e / q_e:.1f}x"
    )
    print()

100%|██████████| 10/10 [00:09<00:00,  1.05it/s]

Hop    Strategy     Nodes (mean±std)          Bridge (mean±std)         Edges (mean±std)
-----------------------------------------------------------------------------------------------
  2    Unified      6472 ± 4160         6416 ± 4136         19996 ± 13164
  2    Q->A only    110 ± 128         53 ± 110         178 ± 353
  2    Ratio        59.1x                                                112.5x

  3    Unified      56873 ± 24619         56816 ± 24595         290027 ± 131663
  3    Q->A only    4414 ± 4314         4358 ± 4312         18295 ± 16176
  3    Ratio        12.9x                                                15.9x



# PubmedQA size growth by K

In [35]:
with open("data/pubmedQA/mapped/test.jsonl", "r") as file:
    dev_data = [json.loads(line) for line in file]

test_data = random.sample(dev_data, 100)

In [ ]:
# full umls
pubmed_stats = defaultdict(lambda: defaultdict(list))

for instance in tqdm(test_data):
    cuis = instance["concept"] or []

    if cuis:
        r = get_unified_khop(cuis, max_hop=2)
        for hop in [1, 2]:
            pubmed_stats[hop]["nodes"].append(r[hop]["cumulative_nodes"])
            pubmed_stats[hop]["bridge"].append(r[hop]["bridge_nodes"])
            pubmed_stats[hop]["edges"].append(r[hop]["cumulative_edges"])

# 打印结果
print(
    f"{'Hop':<6} {'Nodes (mean±std)':<25} {'Bridge (mean±std)':<25} {'Edges (mean±std)'}"
)
print("-" * 95)
for hop in [1, 2]:
    n = pubmed_stats[hop]["nodes"]
    b = pubmed_stats[hop]["bridge"]
    e = pubmed_stats[hop]["edges"]
    print(
        f"  {hop:<4} "
        f"{np.mean(n):.0f} ± {np.std(n):.0f}{'':>8} "
        f"{np.mean(b):.0f} ± {np.std(b):.0f}{'':>8} "
        f"{np.mean(e):.0f} ± {np.std(e):.0f}"
    )
    print()


100%|██████████| 100/100 [19:07<00:00, 11.47s/it]

Hop    Nodes (mean±std)          Bridge (mean±std)         Edges (mean±std)
-----------------------------------------------------------------------------------------------
  1    38569 ± 20713         38495 ± 20709         319408 ± 221057

  2    611280 ± 256967         611206 ± 256961         4976540 ± 1283668



In [ ]:
# dragon
pubmed_stats = defaultdict(lambda: defaultdict(list))

for instance in tqdm(test_data):
    cuis = instance["concept"] or []

    if cuis:
        r = get_unified_khop(cuis, max_hop=2)
        for hop in [1, 2]:
            pubmed_stats[hop]["nodes"].append(r[hop]["cumulative_nodes"])
            pubmed_stats[hop]["bridge"].append(r[hop]["bridge_nodes"])
            pubmed_stats[hop]["edges"].append(r[hop]["cumulative_edges"])

# 打印结果
print(
    f"{'Hop':<6} {'Nodes (mean±std)':<25} {'Bridge (mean±std)':<25} {'Edges (mean±std)'}"
)
print("-" * 95)
for hop in [1, 2]:
    n = pubmed_stats[hop]["nodes"]
    b = pubmed_stats[hop]["bridge"]
    e = pubmed_stats[hop]["edges"]
    print(
        f"  {hop:<4} "
        f"{np.mean(n):.0f} ± {np.std(n):.0f}{'':>8} "
        f"{np.mean(b):.0f} ± {np.std(b):.0f}{'':>8} "
        f"{np.mean(e):.0f} ± {np.std(e):.0f}"
    )
    print()
